Project Title : We are building an AI model that can look at palm oil leaf images and tell whether the plant is healthy or has nutrient problems

In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np


In [3]:
batch_size = 32
image_size = (128, 128)

In [4]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.3,
    shear_range=0.3,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3]
)

val_datagen = ImageDataGenerator(rescale=1./255)  

In [5]:
train_data = train_datagen.flow_from_directory(
    "split_data/train",
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical'
)

val_data = val_datagen.flow_from_directory(
    "split_data/val",
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Found 13726 images belonging to 5 classes.
Found 2722 images belonging to 5 classes.


In [6]:
from sklearn.utils.class_weight import compute_class_weight

class_labels = train_data.classes

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(class_labels),
    y=class_labels
)

class_weights = dict(enumerate(class_weights))
print(class_weights)

{0: 1.483891891891892, 1: 1.1656900212314225, 2: 0.700842481490937, 3: 0.7401455918037206, 4: 1.4486543535620053}


In [7]:
model = models.Sequential()

model.add(layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D(2,2))

model.add(layers.Conv2D(64, (3,3), activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D(2,2))

model.add(layers.Conv2D(128, (3,3), activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D(2,2))

model.add(layers.Flatten())

model.add(layers.Dense(64, activation='relu'))

model.add(layers.Dropout(0.6))

model.add(layers.Dense(train_data.num_classes, activation='softmax'))

In [8]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 126, 126, 32)      896       
                                                                 
 batch_normalization (Batch  (None, 126, 126, 32)      128       
 Normalization)                                                  
                                                                 
 max_pooling2d (MaxPooling2  (None, 63, 63, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 61, 61, 64)        18496     
                                                                 
 batch_normalization_1 (Bat  (None, 61, 61, 64)        256       
 chNormalization)                                                
                                                        

In [9]:
loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

model.compile(
    optimizer='adam',
    loss=loss_fn,
    metrics=['accuracy']
)

In [10]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [11]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    class_weight=class_weights,
    callbacks=[early_stop]
) 

Epoch 1/20


429/429 [==============================] - 297s 684ms/step - loss: 1.5297 - accuracy: 0.3181 - val_loss: 1.4972 - val_accuracy: 0.3031
Epoch 2/20
429/429 [==============================] - 300s 698ms/step - loss: 1.3215 - accuracy: 0.4191 - val_loss: 1.2338 - val_accuracy: 0.5735
Epoch 3/20
429/429 [==============================] - 289s 673ms/step - loss: 1.3068 - accuracy: 0.4400 - val_loss: 1.1865 - val_accuracy: 0.6403
Epoch 4/20
429/429 [==============================] - 278s 648ms/step - loss: 1.2402 - accuracy: 0.4854 - val_loss: 1.0782 - val_accuracy: 0.6539
Epoch 5/20
429/429 [==============================] - 282s 658ms/step - loss: 1.2067 - accuracy: 0.5028 - val_loss: 1.0168 - val_accuracy: 0.8409
Epoch 6/20
429/429 [==============================] - 290s 677ms/step - loss: 1.1924 - accuracy: 0.5141 - val_loss: 0.9787 - val_accuracy: 0.8468
Epoch 7/20
429/429 [==============================] - 279s 649ms/step - loss: 1.1806 - accuracy: 0.5075 - val_loss: 0.9736

In [12]:
model.save("cnn_model.h5")

C:\Users\Asus\AppData\Roaming\Python\Python310\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [13]:
y_true = val_data.classes

y_pred = model.predict(val_data)
y_pred_classes = np.argmax(y_pred, axis=1)

86/86 [==============================] - 19s 220ms/step


In [14]:
confidence = np.max(y_pred, axis=1)

print("Sample confidences:")
print(confidence[:10])

Sample confidences:
[0.43769982 0.29385713 0.2890719  0.45955724 0.3663973  0.34995085
 0.45494267 0.2911103  0.31937882 0.36643073]


In [15]:
from sklearn.metrics import classification_report

class_names = list(val_data.class_indices.keys())

print(classification_report(y_true, y_pred_classes, target_names=class_names))

              precision    recall  f1-score   support

       Boron       0.99      0.95      0.97       366
     Healthy       0.95      0.99      0.97       463
      Kalium       0.86      0.95      0.90       777
          Mg       0.97      0.85      0.91       734
    nitrogen       0.99      0.98      0.99       382

    accuracy                           0.94      2722
   macro avg       0.95      0.95      0.95      2722
weighted avg       0.94      0.94      0.94      2722



In [17]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred_classes)

print("Confusion Matrix:\n")
print(cm)

Confusion Matrix:

[[347  11   8   0   0]
 [  0 458   5   0   0]
 [  2  10 742  21   2]
 [  0   0 107 627   0]
 [  0   4   2   0 376]]
